> **참고용 코드**: sample_data/에는 예시 파일 1개씩만 포함되어 있습니다.


# Climate Embedding Variables (CEV) — PatchTST & Informer

WorldClim 10-arcmin 월별 tmin/tmax/prec (1990-2024, 420개월)에서 각 알고리즘으로
`bio*.asc` 와 동형인 **전세계 CEV `.asc`** 생성 후, bio와의 관계·SDM 성능 비교.


- 차원 k = PCA 누적 설명분산 ≥ 임계값 자동. 학습=subsample, 추론=전체 land 청크.
- 공통 하이퍼파라미터(d_model·layers·dropout·lr·epochs·마스킹)는 두 모델 동일, 아키텍처 고유값만 다름.
- 마스킹 통일: 둘 다 **연속 12개월 구간(span)** 마스킹. (PatchTST=패치 단위가 곧 12개월 span)
- 리소스 자동 감지(GPU/VRAM·CPU)
- 셀: 1)config 2)로딩 3)전처리 4~6)PatchTST 7~9)Informer 10)두 모델 점검 11)**bio 유사도** 12)**SDM 비교(Pja_v4.csv)**.


In [ ]:
# === Cell 1. Config + 리소스 자동 감지 ================================
import os, re, glob, math, time, warnings
import numpy as np
import torch, torch.nn as nn
from concurrent.futures import ThreadPoolExecutor
from sklearn.decomposition import PCA
warnings.filterwarnings("ignore")

BASE   = "WorldClim_10min_1990-2024"
DIRS   = {"tmin": os.path.join(BASE,"wc2.1_cruts4.09_10m_tmin_1990-2024"),
          "tmax": os.path.join(BASE,"wc2.1_cruts4.09_10m_tmax_1990-2024"),
          "prec": os.path.join(BASE,"wc2.1_cruts4.09_10m_prec_1990-2024")}
PREFIX = {"tmin":"wc2.1_cruts4.09_10m_tmin","tmax":"wc2.1_cruts4.09_10m_tmax",
          "prec":"wc2.1_cruts4.09_10m_prec"}
DEM_PATH = "dem.asc"
BIO_DIR  = "bcVariables9024"
OCC_CSV  = "Pja_v4.csv"
OUTDIR   = "CEV_output"; os.makedirs(OUTDIR, exist_ok=True)
CHANNELS = ["tmin","tmax","prec"]; N_MONTHS = 420

# --- 하이퍼파라미터 (공통값은 두 모델 동일 = 공정 비교) ---
VAR_THRESH = 0.95
MAX_K      = 40
D_MODEL, N_HEADS, E_LAYERS, D_FF, DROPOUT = 128, 8, 3, 256, 0.2   # 공통
PATCH_LEN, STRIDE = 12, 6          # PatchTST 고유 (50% overlap -> 69 patch)
INFORMER_FACTOR   = 5              # Informer 고유 (ProbSparse)
MASK_RATIO, MASK_SPAN = 0.40, 12   # 연속 12개월 span 마스킹
EPOCHS = 20
LR     = 5e-4
SEED   = 42
TRAIN_SUBSAMPLE = 50_000          # 학습 land px (None=전체). 추론은 항상 전체.

# --- 리소스 자동 감지 (수동 오버라이드: 아래 변수 직접 지정) ---
N_CPU = os.cpu_count() or 8
torch.set_num_threads(N_CPU)
IO_WORKERS = min(12, N_CPU)
if torch.cuda.is_available():
    DEVICE = "cuda"; _p = torch.cuda.get_device_properties(0); VRAM_GB = _p.total_memory/1e9
    USE_AMP = True
    BATCH = 512
    GPU_NAME = _p.name
else:
    DEVICE, VRAM_GB, USE_AMP, BATCH, GPU_NAME = "cpu", 0.0, False, 256, "-"
INFER_BATCH = BATCH * 2
torch.manual_seed(SEED); np.random.seed(SEED)
print(f"CPU threads={N_CPU}  IO_workers={IO_WORKERS}")
print(f"DEVICE={DEVICE}  GPU={GPU_NAME}  VRAM={VRAM_GB:.1f}GB  AMP={USE_AMP}  BATCH={BATCH}")


In [ ]:
# === Cell 2. 전지구 land 로딩 (dem 마스크 + 병렬 읽기) =================
import rasterio
def sorted_month_files(dirpath, prefix):
    files = glob.glob(os.path.join(dirpath, prefix + "_*.tif"))
    files = sorted(files, key=lambda f: tuple(map(int,
                   re.search(r"_(\d{4})-(\d{2})\.tif$", f).groups())))
    assert len(files) == N_MONTHS, f"{prefix}: {len(files)}"
    return files
file_lists = {c: sorted_month_files(DIRS[c], PREFIX[c]) for c in CHANNELS}

with rasterio.open(DEM_PATH) as dem:
    G_H, G_W = dem.height, dem.width; G_TRANS = dem.transform
    dem_arr = dem.read(1); DEM_NODATA = dem.nodata if dem.nodata is not None else -9999.0
land = np.isfinite(dem_arr) & (dem_arr != DEM_NODATA)
LAND_ROWS, LAND_COLS = np.where(land); n_land = LAND_ROWS.size
print(f"dem {G_H}x{G_W} cellsize {G_TRANS.a:.6f} dem-land={n_land}")

X = np.empty((n_land, N_MONTHS, len(CHANNELS)), dtype="float32")
tasks = [(ci, t, f) for ci, c in enumerate(CHANNELS) for t, f in enumerate(file_lists[c])]
def _fill(task):
    ci, t, f = task
    with rasterio.open(f) as s:
        a = s.read(1); nd = s.nodata
    v = a[LAND_ROWS, LAND_COLS].astype("float32")
    if nd is not None: v[v == nd] = np.nan
    X[:, t, ci] = v
t0 = time.time()
with ThreadPoolExecutor(max_workers=IO_WORKERS) as ex:
    for i, _ in enumerate(ex.map(_fill, tasks), 1):
        if i % 210 == 0: print(f"  {i}/{len(tasks)} rasters")
print(f"read {time.time()-t0:.1f}s")

valid = np.isfinite(X).all(axis=(1, 2))
X = np.ascontiguousarray(X[valid])
VALID_ROWS, VALID_COLS = LAND_ROWS[valid], LAND_COLS[valid]
N = X.shape[0]
print(f"X: {X.shape} ({X.nbytes/1e9:.2f} GB) dropped {n_land-N} NaN px")


In [ ]:
# === Cell 3. 전처리 + Dataset + 헬퍼 + 학습/추출 루틴 =================
ch_mean = X.reshape(-1, X.shape[-1]).mean(0).astype("float32")
ch_std  = (X.reshape(-1, X.shape[-1]).std(0) + 1e-6).astype("float32")
print("ch_mean", ch_mean.round(3), "ch_std", ch_std.round(3))
MEAN = torch.tensor(ch_mean, device=DEVICE); STD = torch.tensor(ch_std, device=DEVICE)
X_t = torch.from_numpy(X)
N, SEQ_LEN, N_CH = X_t.shape

class SeqDS(torch.utils.data.Dataset):
    def __init__(self, t, idx=None): self.t=t; self.idx=idx
    def __len__(self): return self.t.shape[0] if self.idx is None else len(self.idx)
    def __getitem__(self, i): return self.t[i] if self.idx is None else self.t[self.idx[i]]

rng = np.random.default_rng(SEED)
train_idx = rng.choice(N, TRAIN_SUBSAMPLE, replace=False) if (TRAIN_SUBSAMPLE and TRAIN_SUBSAMPLE < N) else None
PIN = (DEVICE == "cuda")
train_loader = torch.utils.data.DataLoader(SeqDS(X_t, train_idx), batch_size=BATCH, shuffle=True, pin_memory=PIN)
eval_loader  = torch.utils.data.DataLoader(SeqDS(X_t), batch_size=INFER_BATCH, shuffle=False, pin_memory=PIN)
print(f"train px={len(SeqDS(X_t, train_idx))} eval px={N}")

def norm(xb): return (xb - MEAN) / STD

def span_mask(B, L, ratio, span, device):
    # 연속 span 구간 마스킹 (True=가림)
    m = torch.zeros(B, L, dtype=torch.bool, device=device)
    ar = torch.arange(span, device=device)
    for _ in range(max(1, int(ratio * L / span))):
        st = torch.randint(0, L - span + 1, (B,), device=device)
        m.scatter_(1, (st[:, None] + ar[None, :]).clamp_(max=L-1), True)
    return m

with open(DEM_PATH) as _f: ASC_HEADER = [next(_f) for _ in range(6)]
def _scatter(v):
    g = np.full((G_H, G_W), -9999.0, dtype="float32"); g[VALID_ROWS, VALID_COLS] = v; return g
def write_asc(path, grid2d):
    with open(path, "w") as f: f.writelines(ASC_HEADER); np.savetxt(f, grid2d, fmt="%.6g")
def write_embedding(emb, algo, k):
    d = os.path.join(OUTDIR, f"CEV_{algo}"); os.makedirs(d, exist_ok=True)
    for b in range(k): write_asc(os.path.join(d, f"{algo}_CEV{b+1:02d}.asc"), _scatter(emb[:, b]))
    print(f"saved {k} .asc -> {d}")

def reduce_to_emb(reps, thresh=VAR_THRESH, max_k=MAX_K):
    reps = (reps - reps.mean(0)) / (reps.std(0) + 1e-6)
    nc = min(max_k, reps.shape[1], reps.shape[0])
    p = PCA(n_components=nc, svd_solver="randomized", random_state=SEED).fit(reps)
    cum = np.cumsum(p.explained_variance_ratio_); k = int(np.searchsorted(cum, thresh) + 1)
    out = p.transform(reps)[:, :k]
    s = np.sign(np.sum(out**3, axis=0)); s[s == 0] = 1   # 부호 규약(왜도): run 간 밴드 부호 일관
    out *= s
    print(f"  k={k} @ {cum[k-1]*100:.1f}% | 누적 {np.round(cum[:k],3)}")
    return out.astype("float32"), k

write_asc(os.path.join(OUTDIR, "cev_valid_mask.asc"), _scatter(np.ones(N, "float32")))

def train_model(model, tag):
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP); model.train()
    for ep in range(EPOCHS):
        tot = 0.0; t0 = time.time()
        for xb in train_loader:
            xb = norm(xb.to(DEVICE, non_blocking=True)); opt.zero_grad(set_to_none=True)
            if USE_AMP:
                with torch.autocast("cuda"): loss = model(xb)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                loss = model(xb); loss.backward(); opt.step()
            tot += loss.item() * xb.shape[0]
        if (ep+1) % 5 == 0 or ep == 0:
            print(f"[{tag}] ep {ep+1:02d}/{EPOCHS} MSE {tot/len(train_loader.dataset):.4f} {time.time()-t0:.1f}s")

@torch.no_grad()
def extract(model, feat_dim):
    model.eval(); reps = np.empty((N, feat_dim), dtype="float32"); pos = 0
    for xb in eval_loader:                       # 최종 산출물은 fp32 (AMP 미사용)
        e = model.embed(norm(xb.to(DEVICE, non_blocking=True))).float().cpu().numpy()
        reps[pos:pos+e.shape[0]] = e; pos += e.shape[0]
    return reps


## PatchTST
채널 독립, 12개월 patch(stride 6, 69개), 연속 패치 마스킹 복원. patch 평균풀링+채널 concat -> PCA(k).

In [ ]:
# === Cell 4. PatchTST 정의 ============================================
class PatchTST(nn.Module):
    def __init__(self, seq_len, n_ch, patch_len, stride, d_model, n_heads, e_layers, d_ff, dropout):
        super().__init__()
        self.patch_len, self.stride, self.n_ch = patch_len, stride, n_ch
        self.n_patch  = (seq_len - patch_len)//stride + 1
        self.proj_in  = nn.Linear(patch_len, d_model)
        self.pos      = nn.Parameter(torch.randn(1, self.n_patch, d_model)*0.02)
        self.mask_tok = nn.Parameter(torch.randn(d_model)*0.02)
        enc = nn.TransformerEncoderLayer(d_model, n_heads, d_ff, dropout, batch_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(enc, e_layers)
        self.recon   = nn.Linear(d_model, patch_len); self.feat_dim = n_ch*d_model
    def patchify(self, x):
        x = x.permute(0,2,1).unfold(-1, self.patch_len, self.stride)
        return x.reshape(x.shape[0]*self.n_ch, self.n_patch, self.patch_len)
    def encode(self, x, mask=None):
        p = self.patchify(x); tok = self.proj_in(p)
        if mask is not None: tok = torch.where(mask.unsqueeze(-1), self.mask_tok, tok)
        return self.encoder(tok + self.pos), p
    def forward(self, x):
        p = self.patchify(x); BC, NP, _ = p.shape
        # 연속 패치(=12개월 span) 마스킹: 패치 축에서 인접 패치 묶음
        mask = torch.rand(BC, NP, device=x.device) < MASK_RATIO
        mask[mask.all(1), 0] = False
        h, target = self.encode(x, mask); pred = self.recon(h)
        return ((pred-target)**2 * mask.unsqueeze(-1)).sum()/(mask.sum()*self.patch_len + 1e-6)
    @torch.no_grad()
    def embed(self, x):
        h,_ = self.encode(x, None); h = h.mean(1)
        return h.reshape(x.shape[0], self.n_ch, -1).reshape(x.shape[0], -1)

patchtst = PatchTST(N_MONTHS, len(CHANNELS), PATCH_LEN, STRIDE, D_MODEL, N_HEADS, E_LAYERS, D_FF, DROPOUT).to(DEVICE)
print(sum(p.numel() for p in patchtst.parameters()), "params")


In [ ]:
# === Cell 5. PatchTST 학습 ============================================
train_model(patchtst, "PatchTST")


In [ ]:
# === Cell 6. PatchTST 임베딩 추출 + 전세계 .asc 저장 ==================
reps = extract(patchtst, patchtst.feat_dim)
emb_patchtst, k_patchtst = reduce_to_emb(reps)
print("PatchTST emb:", emb_patchtst.shape); write_embedding(emb_patchtst, "PatchTST", k_patchtst)


## Informer
다변량 420x3, ProbSparse self-attention 인코더, 연속 span 마스킹 복원. 시간축 평균풀링 -> PCA(k).

In [ ]:
# === Cell 7. Informer 정의 (ProbSparse attention) =====================
class ProbAttention(nn.Module):
    def __init__(self, factor=5, dropout=0.1):
        super().__init__(); self.factor=factor; self.drop=nn.Dropout(dropout)
    def _prob_QK(self, Q, K, sample_k, n_top):
        B,H,L_K,E = K.shape; _,_,L_Q,_ = Q.shape
        K_exp = K.unsqueeze(-3).expand(B,H,L_Q,L_K,E)
        idx = torch.randint(L_K, (L_Q, sample_k), device=K.device)
        K_smp = K_exp[:,:,torch.arange(L_Q).unsqueeze(1), idx, :]
        Q_K_s = torch.matmul(Q.unsqueeze(-2), K_smp.transpose(-2,-1)).squeeze(-2)
        M = Q_K_s.max(-1)[0] - Q_K_s.sum(-1)/L_K; M_top = M.topk(n_top, sorted=False)[1]
        Q_red = Q[torch.arange(B)[:,None,None], torch.arange(H)[None,:,None], M_top, :]
        return torch.matmul(Q_red, K.transpose(-2,-1)), M_top
    def _context(self, V, L_Q):
        return V.mean(-2).unsqueeze(-2).expand(*V.shape[:2], L_Q, V.shape[-1]).clone()
    def _update(self, ctx, V, scores, idx, L_Q):
        B,H = V.shape[0], V.shape[1]; attn = torch.softmax(scores, -1)
        ctx[torch.arange(B)[:,None,None], torch.arange(H)[None,:,None], idx, :] = \
            torch.matmul(self.drop(attn), V).type_as(ctx)
        return ctx
    def forward(self, q, k, v):
        B,L_Q,H,E = q.shape; L_K = k.shape[1]
        q,k,v = q.transpose(2,1), k.transpose(2,1), v.transpose(2,1)
        U = min(self.factor*int(np.ceil(np.log(L_K))), L_K); u = min(self.factor*int(np.ceil(np.log(L_Q))), L_Q)
        scores, idx = self._prob_QK(q, k, U, u); scores = scores/math.sqrt(E)
        ctx = self._update(self._context(v, L_Q), v, scores, idx, L_Q)
        return ctx.transpose(2,1).contiguous()
class AttentionLayer(nn.Module):
    def __init__(self, d_model, n_heads, factor, dropout):
        super().__init__(); self.dk=d_model//n_heads; self.h=n_heads
        self.q=nn.Linear(d_model,d_model); self.k=nn.Linear(d_model,d_model)
        self.v=nn.Linear(d_model,d_model); self.o=nn.Linear(d_model,d_model)
        self.att = ProbAttention(factor=factor, dropout=dropout)
    def forward(self, x):
        B,L,_ = x.shape
        q=self.q(x).view(B,L,self.h,self.dk); k=self.k(x).view(B,L,self.h,self.dk); v=self.v(x).view(B,L,self.h,self.dk)
        return self.o(self.att(q,k,v).view(B,L,-1))
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, factor, dropout):
        super().__init__(); self.att=AttentionLayer(d_model,n_heads,factor,dropout)
        self.n1=nn.LayerNorm(d_model); self.n2=nn.LayerNorm(d_model)
        self.ff=nn.Sequential(nn.Linear(d_model,d_ff), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_ff,d_model))
        self.drop=nn.Dropout(dropout)
    def forward(self, x):
        x = self.n1(x + self.drop(self.att(x))); return self.n2(x + self.drop(self.ff(x)))
class Informer(nn.Module):
    def __init__(self, seq_len, n_ch, d_model, n_heads, e_layers, d_ff, factor, dropout):
        super().__init__(); self.seq_len=seq_len; self.n_ch=n_ch; self.feat_dim=d_model
        self.val_emb = nn.Conv1d(n_ch, d_model, 3, padding=1)
        self.pos = nn.Parameter(torch.randn(1, seq_len, d_model)*0.02)
        self.mask_tok = nn.Parameter(torch.randn(d_model)*0.02)
        self.layers = nn.ModuleList([EncoderLayer(d_model,n_heads,d_ff,factor,dropout) for _ in range(e_layers)])
        self.recon = nn.Linear(d_model, n_ch)
    def encode(self, x, mask=None):
        h = self.val_emb(x.permute(0,2,1)).permute(0,2,1)
        if mask is not None: h = torch.where(mask.unsqueeze(-1), self.mask_tok, h)
        h = h + self.pos
        for lyr in self.layers: h = lyr(h)
        return h
    def forward(self, x):
        B,L,_ = x.shape
        mask = span_mask(B, L, MASK_RATIO, MASK_SPAN, x.device)   # 연속 span
        pred = self.recon(self.encode(x, mask))
        return ((pred-x)**2 * mask.unsqueeze(-1)).sum()/(mask.sum()*self.n_ch + 1e-6)
    @torch.no_grad()
    def embed(self, x): return self.encode(x, None).mean(1)

informer = Informer(N_MONTHS, len(CHANNELS), D_MODEL, N_HEADS, E_LAYERS, D_FF, INFORMER_FACTOR, DROPOUT).to(DEVICE)
print(sum(p.numel() for p in informer.parameters()), "params")

In [ ]:
# === Cell 8. Informer 학습 ============================================
train_model(informer, "Informer")


In [ ]:
# === Cell 9. Informer 임베딩 추출 + 전세계 .asc 저장 ==================
reps = extract(informer, informer.feat_dim)
emb_informer, k_informer = reduce_to_emb(reps)
print("Informer emb:", emb_informer.shape); write_embedding(emb_informer, "Informer", k_informer)


In [ ]:
# === Cell 10. 두 모델 점검 ============================================
print("PatchTST k =", k_patchtst, "| Informer k =", k_informer)
kk = min(k_patchtst, k_informer)
C = np.corrcoef(emb_patchtst[:, :kk].T, emb_informer[:, :kk].T)[:kk, kk:]
print("PatchTST vs Informer |corr| 최대매칭 평균:", round(np.abs(C).max(1).mean(), 3))


## bio 유사도 분석 (1번)
각 CEV 축을 bio 19개 **전부로 회귀 → R²**. R² 높음=bio로 재현됨(중복), 낮음=bio가 못 담는 **새 정보**.
보조로 단일 bio 최대 |상관|. (공간 자기상관 때문에 p값은 신뢰 불가 → 크기만 해석)

In [ ]:
# === Cell 11. CEV vs bio 유사도 (R² + 상관) ===========================
from sklearn.linear_model import LinearRegression
bio_files = sorted(glob.glob(os.path.join(BIO_DIR, "bio*.tif")),
                   key=lambda f: int(re.search(r"bio(\d+)\.tif$", f).group(1)))
assert len(bio_files) == 19, bio_files
BIO = np.stack([rasterio.open(f).read(1) for f in bio_files])      # (19,H,W)
bio_v = BIO[:, VALID_ROWS, VALID_COLS].T.astype("float32")        # (N,19)
fin = np.isfinite(bio_v).all(1) & (bio_v != -9999).all(1)
Bz = bio_v[fin]; Bz = (Bz - Bz.mean(0)) / (Bz.std(0) + 1e-6)
print(f"공통 유효셀 {fin.sum()} / {N}")
for algo, emb in [("PatchTST", emb_patchtst), ("Informer", emb_informer)]:
    E = emb[fin]; r2s = []
    print(f"\n[{algo}]  CEV축: R²(bio 19개 전부)  최대단일|r|")
    for j in range(E.shape[1]):
        r2 = LinearRegression().fit(Bz, E[:, j]).score(Bz, E[:, j])
        c = np.array([abs(np.corrcoef(Bz[:, i], E[:, j])[0, 1]) for i in range(19)])
        r2s.append(r2)
        print(f"  CEV{j+1:02d}:  R²={r2:.2f}   |r|max={c.max():.2f} (bio{c.argmax()+1})")
    print(f"  >> 평균 R²={np.mean(r2s):.2f}  ")


## SDM 성능 비교 (2번)
`Pja_v4.csv` occurrence로 predictor 세트별(bio / CEV-PatchTST / CEV-Informer) 예측성능 비교.
- 셀 단위 thinning(중복 제거), random background, **공간 블록 교차검증**(자기상관 보정).
- 빠른 비교용 **로지스틱(선형+2차항) 프록시** — 확정 수치는 .asc 로 Java MaxEnt 권장.

In [ ]:
# === Cell 12. SDM 비교 (Pja_v4.csv, 공간블록 CV AUC) ==================
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

occ = pd.read_csv(OCC_CSV)
col = ((occ.longitude.values - G_TRANS.c) / G_TRANS.a).astype(int)
row = ((G_TRANS.f - occ.latitude.values) / G_TRANS.a).astype(int)
ok = (row >= 0) & (row < G_H) & (col >= 0) & (col < G_W)
row, col = row[ok], col[ok]
IDX = np.full((G_H, G_W), -1, dtype=np.int64); IDX[VALID_ROWS, VALID_COLS] = np.arange(N)
vi = IDX[row, col]; vi = np.unique(vi[vi >= 0])                    # valid + 셀 thinning
print(f"occurrence: {len(occ)} -> 유효·thinned {vi.size}")

rng2 = np.random.default_rng(SEED)
bg = rng2.choice(np.setdiff1d(np.arange(N), vi), size=min(10000, N - vi.size), replace=False)
cells = np.concatenate([vi, bg]); y = np.concatenate([np.ones(vi.size), np.zeros(bg.size)])
BLK = 30                                                            # 30셀 ≈ 5° 공간블록
groups = (VALID_ROWS[cells] // BLK) * 10000 + (VALID_COLS[cells] // BLK)

def predmat(name):
    if name == "bio":      return BIO[:, VALID_ROWS[cells], VALID_COLS[cells]].T.astype("float32")
    if name == "PatchTST": return emb_patchtst[cells]
    if name == "Informer": return emb_informer[cells]

def cv_auc(Xp):
    Xp = np.concatenate([Xp, Xp**2], axis=1)                       # 선형+2차 (MaxEnt 근사)
    aucs = []; gkf = GroupKFold(n_splits=5)
    for tr, te in gkf.split(Xp, y, groups):
        if len(np.unique(y[te])) < 2: continue
        sc = StandardScaler().fit(Xp[tr])
        m = LogisticRegression(max_iter=1000, C=1.0).fit(sc.transform(Xp[tr]), y[tr])
        aucs.append(roc_auc_score(y[te], m.predict_proba(sc.transform(Xp[te]))[:, 1]))
    return np.mean(aucs), np.std(aucs)

print("\npredictor set   n_feat   spatial-block CV AUC")
for name in ["bio", "PatchTST", "Informer"]:
    Xp = predmat(name); mfin = np.isfinite(Xp).all(1)
    if not mfin.all(): Xp = np.nan_to_num(Xp, nan=0.0)
    mu, sd = cv_auc(Xp)
    print(f"  {name:10s}   {Xp.shape[1]:>4d}     AUC = {mu:.3f} ± {sd:.3f}")
print("")
print("주의: background=전지구 -> AUC 부풀 수 있음(상대비교·sanity용; v2 실질판단은 Cell 11). "
      "k 차이=n_feat 영향. 확정수치는 .asc -> MaxEnt")
